In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

In [ ]:
from pathlib import Path

from src.services.gec.config import (
    CHECKPOINT_PATH,
    ID2LABEL_PATH,
    LABEL2ID_PATH,
    NOPNX_TRAIN_OUTPUT,
    PNX_TRAIN_OUTPUT,
    TRAIN_COR_PATH,
    TRAIN_SENT_PATH,
)
from src.services.gec.features.common import build_dataset_builder
from src.services.gec.features.exporter import DatasetExporter
from src.services.gec.features.feature_builder import DatasetBuilder
from src.services.gec.features.pruner import LabelPruner
from src.services.gec.features.vocabulary import LabelVocabularyBuilder
from src.services.gec.modules.edit_tagger.preprocessing.segregator import Segregator

In [ ]:
import json


def save_json(
    data: dict,
    output_path: Path,
) -> None:
    """Save data dict to a JSON file."""
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(
            data,
            f,
            ensure_ascii=False,
            indent=2,
        )

In [ ]:
def build_train() -> None:
    """Build the training dataset."""
    builder: DatasetBuilder = build_dataset_builder()  # noqa: F821

    examples = builder.build_pipeline(TRAIN_SENT_PATH, TRAIN_COR_PATH, CHECKPOINT_PATH)

    pruner = LabelPruner(min_frequency=0)
    examples = pruner.prune(examples)
    print("examples: ", examples)

    segregator = Segregator()
    nopnx_examples, pnx_examples = segregator.segregate(examples)

    vocab_builder = LabelVocabularyBuilder()

    label2id, id2label = vocab_builder.build(examples)

    save_json(label2id, LABEL2ID_PATH)
    save_json(id2label, ID2LABEL_PATH)

    exporter = DatasetExporter()

    exporter.export_jsonl(
        nopnx_examples,
        NOPNX_TRAIN_OUTPUT,
    )

    exporter.export_jsonl(
        pnx_examples,
        PNX_TRAIN_OUTPUT,
    )

In [ ]:
# from pathlib import Path
# import shutil

# BASE_DIR = Path("/home/somia/baligh")

# lab_file = BASE_DIR
# "src
# services
# gec
# data
# edit_tagger
# processed
# lab.jsonl"
# tokens_file = BASE_DIR
# "src
# services
# gec
# data
# edit_tagger
# processed
# tokens_labels.jsonl"
# output_file = BASE_DIR
# "src
# services
# gec
# data
# edit_tagger
# processed
# combined.jsonl"

# with output_file.open("w", encoding="utf-8") as out:
#     for file in (lab_file, tokens_file):
#         with file.open("r", encoding="utf-8") as f:
#             shutil.copyfileobj(f, out)


In [ ]:
from pathlib import Path

from src.services.gec.modules.edit_tagger.common import ProjectedExample


def load_projected_examples(
    path: str,
) -> list[ProjectedExample]:
    """Load projected examples from a JSONL file."""
    results: list[ProjectedExample] = []

    with Path(path).open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            data = json.loads(line)

            results.append(ProjectedExample(**data))

    return results

In [ ]:
exist = True
if not exist:
    examples = builder.build_pipeline(TRAIN_SENT_PATH, TRAIN_COR_PATH, CHECKPOINT_PATH)  # noqa: F821

In [ ]:
examples = load_projected_examples(
    "/home/somia/baligh/src/services/gec/data/edit_tagger/processed/tokens_labels.jsonl"
)
print(len(examples))

In [ ]:
print(len(examples))
pruner = LabelPruner(min_frequency=3)
pruned_examples = pruner.prune(examples)
print("examples: ", len(pruned_examples))

In [ ]:
vocab_builder = LabelVocabularyBuilder()

label2id, id2label = vocab_builder.build(pruned_examples)
print(label2id, id2label)

In [ ]:
save_json(label2id, LABEL2ID_PATH)
save_json(id2label, ID2LABEL_PATH)

In [ ]:
segregator = EditSegregator()
nopnx_examples, pnx_examples = segregator.simple_segregate(examples)
print(pnx_examples)

In [ ]:
exporter.export_jsonl(  # noqa: F821
    nopnx_examples,
    NOPNX_TRAIN_OUTPUT,
)

exporter.export_jsonl(  # noqa: F821
    pnx_examples,
    PNX_TRAIN_OUTPUT,
)